# Practical Exam: Automating Customer Support with OpenAI API

You work as an AI Engineer at ChatSolveAI, a company that provides automated customer support solutions. The company wants to improve response times and accuracy in answering customer queries by leveraging OpenAI’s GPT models.

Your task is to build a chatbot that classifies customer queries, retrieves relevant responses, and logs interactions in a structured way. The chatbot will use text embeddings, similarity search, API calls, and conversation management techniques.


**Please note:** 

1. The OpenAI Embeddings API supports passing a list of strings to the input parameter in a single request. This allows you to generate multiple embeddings at once without looping over individual elements, which can significantly improve efficiency and reduce the risk of hitting rate limits.

2. When submitting your solution, you may see an error message reading 'Something went wrong while submitting your solution. Please try again.' This is because using the OpenAI API may mean code takes longer to run than code in our other Certifications. Please ignore this message if your code is taking a few minutes to run. However, if your code makes too many API requests, the API will time out. If your cells run for more than a few minutes each, you may need to consider revising your code. 

In [ ]:
S# Run this cell before running your solution

# Import necessary modules
import os
from openai import OpenAI

# Define the model to use
model = "gpt-3.5-turbo"

# Define the client
client = OpenAI()

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

# Task 1

ChatSolveAI has provided a knowledge base (`knowledge_base.csv`) containing information about various products, services, and customer policies. To enhance search and query capabilities, you need to convert this data into embeddings and store them for efficient retrieval.

- Load the dataset (`knowledge_base.csv`).
- Generate text embeddings using OpenAI’s embedding model (`text-embedding-3-small`). Each document's `document_text` should be transformed into an embedding vector. Do not apply any text transformations such as lowercasing, stripping or normalization before embedding.
- Store the generated embeddings in a structured format (`knowledge_embeddings.json`) with the following format available below.
- Store the embedded data and associated metadata for retrieval.  

### Format to store generated embeddings:
```json
[
    {
       "document_id": 1,
       "document_text": "Example document text.",
       "embedding_vector": [0.123, 0.456, ...],
       "metadata": "Additional document info"
    }
]
```

### Data description: 

| Column Name       | Criteria                                                |
|-------------------|---------------------------------------------------------|
| document_id       | Integer. Unique identifier for each document. No missing values. |
| document_text     | String. Text content of the knowledge base. Preprocessed and embedded. |
| embedding_vector  | List. Embedding representation of the `document_text`. |
| metadata          | String. Metadata for additional information. |


In [ ]:
# Task 1: Generate and store embeddings for the knowledge base
import json
import time
import pandas as pd

knowledge_df = pd.read_csv("knowledge_base.csv")

document_texts = knowledge_df["document_text"].tolist()


def create_embeddings_with_retry(texts, embed_model="text-embedding-3-small", max_retries=5):
    """Call the OpenAI Embeddings API with exponential-backoff retries."""
    for attempt in range(max_retries):
        try:
            response = client.embeddings.create(model=embed_model, input=texts)
            return [item.embedding for item in response.data]
        except Exception as err:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


embedding_vectors = create_embeddings_with_retry(document_texts)

knowledge_embeddings = [
    {
        "document_id": int(row["document_id"]),
        "document_text": row["document_text"],
        "embedding_vector": embedding_vectors[i],
        "metadata": row["metadata"],
    }
    for i, row in knowledge_df.iterrows()
]

with open("knowledge_embeddings.json", "w") as f:
    json.dump(knowledge_embeddings, f)

print(f"Stored {len(knowledge_embeddings)} document embeddings in knowledge_embeddings.json")

# Task 2

ChatSolveAI receives customer queries that need to be classified and matched with appropriate responses. Your task is to preprocess and embed these queries, perform similarity searches on predefined responses (contained in `predefined_responses.json`), and retrieve the most relevant responses.

- Load the dataset (`processed_queries.csv`).
- Retrieve responses by using cosine similarity to perform a similarity search against predefined responses in `predefined_responses.json`.
- Structure API requests properly and implement error handling, including retry mechanisms to handle rate limits.
- Format model responses as JSON to maintain consistency in output.
- Compute confidence scores for retrieved responses, scaled to 0-1.
- Store the structured responses in a JSON file (`query_responses.json`), suitable for integration with other applications. Your JSON file should be structured as follows:

| Column Name       | Criteria                                                   |
|-------------------|------------------------------------------------------------|
| query_id         | Integer. Unique identifier for each query. No missing values. |
| query_text       | String. Preprocessed query text. |
| top_responses    | List. Top 3 most relevant responses retrieved. |
| confidence_scores | List. Model-based confidence score for the top 3 responses. |

In [ ]:
# Task 2: Match customer queries to predefined responses via cosine similarity
import json
import time
import numpy as np
import pandas as pd

queries_df = pd.read_csv("processed_queries.csv")

with open("predefined_responses.json", "r") as f:
    predefined_responses = json.load(f)


QUESTION_KEYS = ("query_text", "query", "question", "intent", "category", "label")
ANSWER_KEYS = ("retrieved_response", "response_text", "response", "answer", "reply", "message", "text", "content")


def _pick_str(item, keys):
    for key in keys:
        if isinstance(item, dict) and key in item and isinstance(item[key], str):
            return item[key]
    return None


def _extract_pair_from_dict(item):
    """Return (match_text, return_text) from a dict entry."""
    answer = _pick_str(item, ANSWER_KEYS)
    question = _pick_str(item, QUESTION_KEYS)
    if answer is not None and question is not None:
        return question, answer
    if answer is not None:
        return answer, answer
    if question is not None:
        return question, question
    text = str(item)
    return text, text


def normalize_predefined(data):
    """Normalize predefined_responses.json into a list of (match_text, return_text) pairs.

    Supports:
    - dict mapping label/category -> response text (or nested dict)
    - list of dicts (Q&A pairs or response objects)
    - list of strings
    """
    pairs = []
    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, str):
                pairs.append((value, value))
            elif isinstance(value, dict):
                q = _pick_str(value, QUESTION_KEYS) or (key if isinstance(key, str) else str(key))
                a = _pick_str(value, ANSWER_KEYS) or str(value)
                pairs.append((q, a))
            elif isinstance(value, list) and value and isinstance(value[0], str):
                response_text = value[0]
                pairs.append((response_text, response_text))
            else:
                pairs.append((str(key), str(value)))
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, str):
                pairs.append((item, item))
            elif isinstance(item, dict):
                pairs.append(_extract_pair_from_dict(item))
            else:
                pairs.append((str(item), str(item)))
    else:
        pairs.append((str(data), str(data)))
    return pairs


print("predefined_responses container type:", type(predefined_responses).__name__)
if isinstance(predefined_responses, dict):
    sample_k = next(iter(predefined_responses))
    print("Sample key:", sample_k, "| sample value:", predefined_responses[sample_k])
elif isinstance(predefined_responses, list) and predefined_responses:
    print("Sample item:", predefined_responses[0])

pairs = normalize_predefined(predefined_responses)
question_texts = [p[0] for p in pairs]
answer_texts = [p[1] for p in pairs]
print(f"Normalized {len(pairs)} predefined responses for matching.")


def embed_texts(texts, embed_model="text-embedding-3-small", max_retries=5, batch_size=500):
    """Batch embed texts with retries and exponential backoff for rate limits/errors."""
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        for attempt in range(max_retries):
            try:
                resp = client.embeddings.create(model=embed_model, input=batch)
                embeddings.extend([item.embedding for item in resp.data])
                break
            except Exception:
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)
    return embeddings


response_embeddings = np.array(embed_texts(question_texts))

query_texts = queries_df["query_text"].tolist()
query_embeddings = np.array(embed_texts(query_texts))


def cosine_similarity_matrix(a, b):
    """Row-wise cosine similarity between matrix a and matrix b."""
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a_norm @ b_norm.T


similarity_matrix = cosine_similarity_matrix(query_embeddings, response_embeddings)

query_responses = []
for i, row in queries_df.iterrows():
    sims = similarity_matrix[i]
    top_idx = np.argsort(sims)[::-1][:3]
    top_responses = [answer_texts[j] for j in top_idx]
    confidence_scores = [float(np.clip(sims[j], 0.0, 1.0)) for j in top_idx]

    query_responses.append(
        {
            "query_id": int(row["query_id"]),
            "query_text": row["query_text"],
            "top_responses": top_responses,
            "confidence_scores": confidence_scores,
        }
    )

with open("query_responses.json", "w") as f:
    json.dump(query_responses, f, indent=2)

print(f"Stored {len(query_responses)} query responses in query_responses.json")
print("Sample entry:", json.dumps(query_responses[0], indent=2))

# Task 3

To provide seamless customer service, ChatSolveAI wants to develop a chatbot that can respond to customer queries efficiently by searching for relevant responses and generating new ones when necessary.

- Develop a chatbot that:
    - Accepts customer queries via text input.
    - Searches for the most relevant responses from a predefined set of responses (`chatbot_responses.json`).
    - Uses the OpenAI Embeddings API (`text-embedding-3-small`) to compute semantic similarity between queries.
    - If no relevant response is found from the predefined set, generates a new response using GPT-3.5-turbo.
- Stores conversation history, including:
    - Query text
    - Retrieved response
    - Timestamp of the interaction
    - Confidence score of the response
- Include one open-ended query not in the predefined responses (e.g., about the refund policy) to test the chatbot’s ability to handle unmatched queries.
- Include one paraphrased query about support hours (e.g., “When can I talk to someone from support?”) to test semantic similarity matching.
- Store structured chatbot responses in a JSON file (`sample_chatbot_responses.json`). Make sure they follow this format:
```json
[
    {
        "query_text": "How do I reset my password?",
        "retrieved_response": "You can reset your password by clicking 'Forgot Password' on the login page.",
        "timestamp": "2025-04-02T14:30:00Z",
        "confidence_score": 0.92
    },
    {
        "query_text": "What are your business hours?",
        "retrieved_response": "Our support team is available from 9 AM to 5 PM, Monday to Friday.",
        "timestamp": "2025-04-02T14:35:00Z",
        "confidence_score": 0.87
    }
]
```

In [ ]:
# Task 3: Chatbot with semantic retrieval and GPT-3.5-turbo fallback
import json
import time
import numpy as np
from datetime import datetime, timezone

with open("chatbot_responses.json", "r") as f:
    chatbot_responses = json.load(f)


_QUESTION_KEYS = ("query_text", "query", "question")
_ANSWER_KEYS = ("retrieved_response", "response_text", "response", "answer", "text", "content")


def _pick_str(item, keys):
    for key in keys:
        if isinstance(item, dict) and key in item and isinstance(item[key], str):
            return item[key]
    return None


def _extract_qa(item):
    """Return (question_for_matching, answer_to_return) from a predefined chatbot entry."""
    question = _pick_str(item, _QUESTION_KEYS)
    answer = _pick_str(item, _ANSWER_KEYS)
    if question is not None and answer is not None:
        return question, answer
    if answer is not None:
        return answer, answer
    if question is not None:
        return question, question
    text = str(item)
    return text, text


_qa_pairs = [_extract_qa(r) for r in chatbot_responses]
predefined_questions = [q for q, _ in _qa_pairs]
predefined_answers = [a for _, a in _qa_pairs]


def _embed(texts, embed_model="text-embedding-3-small", max_retries=5):
    """Embed text(s) with retry/backoff. Accepts a string or list of strings."""
    for attempt in range(max_retries):
        try:
            resp = client.embeddings.create(model=embed_model, input=texts)
            return [item.embedding for item in resp.data]
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


predefined_embeddings = np.array(_embed(predefined_questions))


def _cosine(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


conversation_history = []

SYSTEM_PROMPT = (
    "You are a helpful customer support assistant for ChatSolveAI. "
    "Provide concise, accurate responses to customer queries. "
    "Use the prior conversation for context when relevant."
)


def _build_messages(query_text, history, max_turns=5):
    """Build a chat messages list that includes the recent conversation history as context."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history[-max_turns:]:
        messages.append({"role": "user", "content": turn["query_text"]})
        messages.append({"role": "assistant", "content": turn["retrieved_response"]})
    messages.append({"role": "user", "content": query_text})
    return messages


def chatbot(query_text, similarity_threshold=0.75):
    """Answer a customer query: retrieve from predefined responses, else fall back to GPT-3.5-turbo.

    Conversation history is appended after each turn and is passed back into the GPT fallback
    so the chatbot can maintain multi-turn context.
    """
    query_embedding = _embed(query_text)[0]

    similarities = [_cosine(query_embedding, r_emb) for r_emb in predefined_embeddings]
    best_idx = int(np.argmax(similarities))
    best_sim = float(similarities[best_idx])
    confidence_score = float(np.clip(best_sim, 0.0, 1.0))

    if best_sim >= similarity_threshold:
        retrieved_response = predefined_answers[best_idx]
    else:
        messages = _build_messages(query_text, conversation_history)
        for attempt in range(5):
            try:
                completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
                retrieved_response = completion.choices[0].message.content.strip()
                break
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)

    interaction = {
        "query_text": query_text,
        "retrieved_response": retrieved_response,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "confidence_score": confidence_score,
    }
    conversation_history.append(interaction)
    return interaction


def reset_conversation():
    """Clear the in-memory conversation history (useful between sessions)."""
    conversation_history.clear()


sample_queries = [
    "When can I talk to someone from support?",
    "What is your refund policy for orders placed over 60 days ago?",
]

sample_chatbot_responses = [chatbot(q) for q in sample_queries]

with open("sample_chatbot_responses.json", "w") as f:
    json.dump(sample_chatbot_responses, f, indent=2)

print(f"Stored {len(sample_chatbot_responses)} chatbot interactions in sample_chatbot_responses.json")
for entry in sample_chatbot_responses:
    print(json.dumps(entry, indent=2))